In [1]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

kategoriler = [
    "rec.sport.hockey",
    "sci.space",
    "comp.graphics",
    "talk.politics.misc",
]

egitim = fetch_20newsgroups(
    subset="train",
    categories=kategoriler,
    remove=("headers", "footers", "quotes"),
    random_state=42,
)
test = fetch_20newsgroups(
    subset="test",
    categories=kategoriler,
    remove=("headers", "footers", "quotes"),
    random_state=42,
)

print("Train:", len(egitim.data), "Test:", len(test.data))
print(egitim.target_names)
print(pd.Series(egitim.target).map(lambda i: egitim.target_names[i]).value_counts())
print()
print(egitim.data[0][:500])

Train: 2242 Test: 1492
['comp.graphics', 'rec.sport.hockey', 'sci.space', 'talk.politics.misc']
rec.sport.hockey      600
sci.space             593
comp.graphics         584
talk.politics.misc    465
Name: count, dtype: int64

Pascal Perret, in article <1993Apr21.125750.263@eicn.etna.ch>, wrote

Funny thing, the InterNet: I have no idea what xv221 might be - except that it
might be something to do with X-windows on PCs (? If you know, and have used
it, and think that it is good, email me. [ryanph@mrl.dsto.gov.au]).

DV/X is a common abbreviation for QuarterDeck corporation's Desqview/X
software.

I have not used DV/X yet, but reading the blurbs that Quarterdeck sent me, it
sounds pretty great:
	* allows multiple DOS m


In [2]:
from sklearn.model_selection import train_test_split

X_train_metin, X_val_metin, y_train, y_val = train_test_split(
    egitim.data,
    egitim.target,
    test_size=0.2,
    stratify=egitim.target,
    random_state=42,
)

X_test_metin = test.data
y_test = test.target

print("Train:", len(X_train_metin), "Val:", len(X_val_metin), "Test:", len(X_test_metin))
print("Val dağılımı:", pd.Series(y_val).value_counts(normalize=True).sort_index().round(3).to_dict())

Train: 1793 Val: 449 Test: 1492
Val dağılımı: {0: 0.261, 1: 0.267, 2: 0.265, 3: 0.207}


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", min_df=2)

X_train = vectorizer.fit_transform(X_train_metin)
X_val = vectorizer.transform(X_val_metin)
X_test = vectorizer.transform(X_test_metin)

print("X_train boyutu:", X_train.shape)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

val_pred = model.predict(X_val)
print("İlk 10 tahmin :", val_pred[:10])
print("İlk 10 gerçek :", y_val[:10])

X_train boyutu: (1793, 12401)
İlk 10 tahmin : [2 3 2 1 0 0 3 0 2 0]
İlk 10 gerçek : [2 1 2 1 0 0 3 0 2 0]


In [4]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", round(accuracy_score(y_val, val_pred), 4))
print()
print(classification_report(y_val, val_pred, target_names=egitim.target_names))
print(confusion_matrix(y_val, val_pred))

Accuracy: 0.8953

                    precision    recall  f1-score   support

     comp.graphics       0.92      0.92      0.92       117
  rec.sport.hockey       0.95      0.88      0.91       120
         sci.space       0.85      0.90      0.87       119
talk.politics.misc       0.87      0.87      0.87        93

          accuracy                           0.90       449
         macro avg       0.90      0.89      0.89       449
      weighted avg       0.90      0.90      0.90       449

[[108   1   6   2]
 [  4 106   6   4]
 [  4   2 107   6]
 [  2   3   7  81]]


In [5]:
import numpy as np

yanlis = np.where(val_pred != y_val)[0]
print("Yanlış tahmin sayısı:", len(yanlis))

politika_uzay = np.where((y_val == 3) & (val_pred == 2))[0]
print("Politika mesajına 'uzay' denen sayı:", len(politika_uzay))
print()

for i in politika_uzay[:5]:
    print("Gerçek:", egitim.target_names[y_val[i]], "| Tahmin:", egitim.target_names[val_pred[i]])
    print(X_val_metin[i][:300].replace("\n", " "))
    print("-" * 60)

Yanlış tahmin sayısı: 47
Politika mesajına 'uzay' denen sayı: 7

Gerçek: talk.politics.misc | Tahmin: sci.space
  
------------------------------------------------------------
Gerçek: talk.politics.misc | Tahmin: sci.space
Agreed. Agreed.   Agreed.   And this demonstrates, I assume, that you're a liberal. :-).
------------------------------------------------------------
Gerçek: talk.politics.misc | Tahmin: sci.space
_TOO_ many.
------------------------------------------------------------
Gerçek: talk.politics.misc | Tahmin: sci.space
 Fishing rights are disputed.  Between 2 nations, no matter *how*  friendly, there is ALWAYS fishing disputes.  What I was getting at was the 54 40' or fight slogan is OLD STUFF dealing with the LAND dispute.  No one is saying 54 40' or fight  about fishing rights.  The territorial dispute about the
------------------------------------------------------------
Gerçek: talk.politics.misc | Tahmin: sci.space
 Only the most comprehensive survey on sexuality in

In [6]:
bos = vectorizer.transform([""])

print("Tahmin:", egitim.target_names[model.predict(bos)[0]])
print("Olasılıklar:", dict(zip(egitim.target_names, model.predict_proba(bos)[0].round(3))))

Tahmin: sci.space
Olasılıklar: {'comp.graphics': np.float64(0.254), 'rec.sport.hockey': np.float64(0.246), 'sci.space': np.float64(0.286), 'talk.politics.misc': np.float64(0.214)}


In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

modeller = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42),
}

for ad, m in modeller.items():
    pipe = make_pipeline(
        TfidfVectorizer(lowercase=True, stop_words="english", min_df=2),
        m,
    )
    skorlar = cross_val_score(pipe, X_train_metin, y_train, cv=5, scoring="f1_macro")
    print(f"{ad:22s} F1(macro) ort={skorlar.mean():.3f} sapma={skorlar.std():.3f}")

Logistic Regression    F1(macro) ort=0.896 sapma=0.011
Naive Bayes            F1(macro) ort=0.892 sapma=0.015
Linear SVM             F1(macro) ort=0.897 sapma=0.008


In [8]:
test_pred = model.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, test_pred), 4))
print()
print(classification_report(y_test, test_pred, target_names=egitim.target_names))
print(confusion_matrix(y_test, test_pred))

Test accuracy: 0.8827

                    precision    recall  f1-score   support

     comp.graphics       0.92      0.90      0.91       389
  rec.sport.hockey       0.97      0.91      0.94       399
         sci.space       0.78      0.90      0.83       394
talk.politics.misc       0.88      0.80      0.84       310

          accuracy                           0.88      1492
         macro avg       0.89      0.88      0.88      1492
      weighted avg       0.89      0.88      0.88      1492

[[352   2  30   5]
 [  4 364  20  11]
 [ 21   2 353  18]
 [  4   9  49 248]]
